# 00 — Install & Setup

Install the `myllm` package and verify the public API surface.

> **Colab users:** Run the setup cell → wait for "Done!" → **Runtime → Restart runtime** → continue from cell 2.

In [ ]:
import subprocess, sys, os

def _is_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

if _is_colab():
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install',
         '--force-reinstall', '--no-cache-dir',
         'git+https://github.com/silvaxxx1/MyLLM.git'],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        print('Install FAILED:', r.stderr[-2000:])
    else:
        print('Done! Now go to Runtime → Restart runtime, then continue.')
else:
    root = os.path.abspath(os.path.join(os.getcwd(), '..'))
    has_uv = subprocess.run(['which', 'uv'], capture_output=True).returncode == 0
    cmd = ['uv', 'pip', 'install', '-e', root] if has_uv else [sys.executable, '-m', 'pip', 'install', '-e', root]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print('Install FAILED:', r.stderr[-2000:])
    else:
        print('Local editable install ready.')


Local editable install ready.


## 1. Verify the package loads

In [2]:
import myllm
print('myllm version :', myllm.__version__)
print('public exports:', myllm.__all__)

2026-05-09 20:08:42.572001: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


myllm version : 0.1.0
public exports: ['LLM', 'GPT', 'ModelConfig', 'GenerationConfig', 'SFTTrainer', 'DPOTrainer', 'PPOTrainer', 'PretrainTrainer', 'SFTTrainerConfig', 'DPOTrainerConfig', 'PPOTrainerConfig', 'TrainerConfig', 'get_tokenizer', 'GPT2Tokenizer', 'LLaMA2Tokenizer', 'LLaMA3Tokenizer', 'train', 'tokenizers', 'configs', 'utils', '__version__']


## 2. Import styles — three ways to access the same API

In [3]:
# Style 1 — flat top-level (most convenient)
from myllm import LLM, ModelConfig, GenerationConfig
from myllm import SFTTrainer, SFTTrainerConfig
from myllm import get_tokenizer 

print('LLM             :', LLM)
print('ModelConfig     :', ModelConfig)
print('GenerationConfig:', GenerationConfig)
print('SFTTrainer      :', SFTTrainer)

LLM             : <class 'myllm.api.LLM'>
ModelConfig     : <class 'myllm.Configs.ModelConfig.ModelConfig'>
GenerationConfig: <class 'myllm.Configs.GenConfig.GenerationConfig'>
SFTTrainer      : <class 'myllm.Train.sft_trainer.SFTTrainer'>


In [4]:
# Style 2 — submodule imports (HuggingFace style)
from myllm.train import SFTTrainer, DPOTrainer, PPOTrainer
from myllm.tokenizers import GPT2Tokenizer
from myllm.configs import ModelConfig, GenerationConfig

print('from myllm.train      :', SFTTrainer)
print('from myllm.tokenizers :', GPT2Tokenizer)
print('from myllm.configs    :', ModelConfig)

from myllm.train      : <class 'myllm.Train.sft_trainer.SFTTrainer'>
from myllm.tokenizers : <class 'myllm.Tokenizers.gpt2_tokenizer.GPT2Tokenizer'>
from myllm.configs    : <class 'myllm.Configs.ModelConfig.ModelConfig'>


In [5]:
# Style 3 — submodule attribute access
import myllm 
print('myllm.train.SFTTrainer        :', myllm.train.SFTTrainer)
print('myllm.tokenizers.GPT2Tokenizer:', myllm.tokenizers.GPT2Tokenizer)

myllm.train.SFTTrainer        : <class 'myllm.Train.sft_trainer.SFTTrainer'>
myllm.tokenizers.GPT2Tokenizer: <class 'myllm.Tokenizers.gpt2_tokenizer.GPT2Tokenizer'>


## 3. CLI

In [6]:
!python -m myllm version
!python -m myllm models

2026-05-09 20:11:20.646874: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
myllm 0.1.0
2026-05-09 20:12:39.742526: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Available model configs:
  gpt2-small
  gpt2-medium
  gpt2-large
  gpt2-xl
  llama2-7b
  llama2-13b
  llama3-1b
  llama3-3b
  llama3-8b


## 4. Available model configs with memory estimates

In [7]:
import torch
from myllm import ModelConfig

print(f'{"Model":<16}  {"Layers":>6}  {"Embd":>5}  {"Params":>8}  {"FP32 (GB)":>9}  {"FP16 (GB)":>9}')
print('-' * 65)
for name in ModelConfig.available_configs():
    cfg  = ModelConfig.from_name(name)
    fp32 = cfg.estimate_memory(dtype=torch.float32)
    fp16 = cfg.estimate_memory(dtype=torch.float16)
    print(f'{name:<16}  {cfg.n_layer:>6}  {cfg.n_embd:>5}  '
          f'{fp32["n_parameters"]/1e6:>7.1f}M  '
          f'{fp32["parameters_gb"]:>9.2f}  {fp16["parameters_gb"]:>9.2f}')

Model             Layers   Embd    Params  FP32 (GB)  FP16 (GB)
-----------------------------------------------------------------
gpt2-small            12    768    151.9M       0.57       0.28
gpt2-medium           24   1024    454.2M       1.69       0.85
gpt2-large            36   1280   1008.2M       3.76       1.88
gpt2-xl               48   1600   2046.8M       7.62       3.81
llama2-7b             32   4096   8721.5M      32.49      16.25
llama2-13b            40   5120  16941.9M      63.11      31.56
llama3-1b             24   2048   1873.5M       6.98       3.49
llama3-3b             32   3072   5226.2M      19.47       9.73
llama3-8b             32   4096   9115.8M      33.96      16.98


In [ ]:
from myllm import LLM, GenerationConfig

llm = LLM.from_pretrained('gpt2-small')
print(llm)

PROMPT = 'Scientists recently discovered that'

def gen(cfg: GenerationConfig) -> str:
    return llm.generate_text(PROMPT, generation_config=cfg, skip_prompt=True)['text']

print('Ready.')


📋 Using custom config
📊 Estimated memory: 0.71 GB
   Parameters: 0.57 GB
   Activations: 0.14 GB
⬇️  Downloading gpt2-small...
✅ File ./models/model-gpt2-small.safetensors already exists and is valid.
📂 Loading weights from disk...
📦 Loading weights from ./models/model-gpt2-small.safetensors to cpu...
✅ Loaded 160 tensors
🏗️  Initializing model architecture...
🔧 Converting model to torch.float32...
🔄 Mapping weights using gpt2_mapper...
Loading gpt2-small weights |

Loading GPT-2 blocks:   0%|          | 0/12 [00:00<?, ?block/s]

Loading gpt2-small weights /

Loading GPT-2 blocks:  25%|██▌       | 3/12 [00:00<00:00, 27.31block/s]

Loading gpt2-small weights -

...

Loading gpt2-small weights \

Loading GPT-2 blocks: 100%|██████████| 12/12 [00:00<00:00, 34.26block/s]

Loading gpt2-small weights |

✅ Successfully loaded gpt2-small!
🎯 Model ready on cuda for inference!
LLM(model='gpt2-small', params=163.0M, device='cuda', dtype=torch.float32)
Ready.


: 